<a href="https://colab.research.google.com/github/DharaCS23181/Skincare_Product_Prediction/blob/main/Random_Forest(80_20).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import tensorflow as tf

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/ML/skin_recommendation_dataset.csv")

In [3]:
import pandas as pd
import numpy as np
import re

from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

# ==============================
# 1. CLEAN FUNCTION
# ==============================

def clean_text(x):
    x = str(x)
    x = re.sub(r"[^a-zA-Z0-9,\-\s]", "", x)
    return [i.strip().lower() for i in x.split(",") if i.strip() != ""]


In [4]:

data = df[['skintype','skin_condition','notable_effects','product_type']].copy()

data['skintype'] = data['skintype'].apply(clean_text)
data['skin_condition'] = data['skin_condition'].apply(clean_text)
data['notable_effects'] = data['notable_effects'].apply(clean_text)


In [5]:

mlb_skin = MultiLabelBinarizer()
mlb_condition = MultiLabelBinarizer()
mlb_effects = MultiLabelBinarizer()

skin_features = pd.DataFrame(
    mlb_skin.fit_transform(data['skintype']),
    columns=mlb_skin.classes_
)

condition_features = pd.DataFrame(
    mlb_condition.fit_transform(data['skin_condition']),
    columns=mlb_condition.classes_
)

effects_features = pd.DataFrame(
    mlb_effects.fit_transform(data['notable_effects']),
    columns=mlb_effects.classes_
)

In [6]:
X = pd.concat([skin_features, condition_features, effects_features], axis=1)

le = LabelEncoder()
y = le.fit_transform(data['product_type'])

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [14]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1
)

grid.fit(X_train, y_train)

rf = grid.best_estimator_

rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, min_samples_leaf=2, min_samples_split=5,
                       n_estimators=200, random_state=42)

In [15]:

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Weighted F1:", f1_score(y_test, y_pred, average='weighted'))
print(classification_report(y_test, y_pred))

Accuracy: 0.5510204081632653
Weighted F1: 0.5484822169362287
              precision    recall  f1-score   support

           0       0.55      0.45      0.49        40
           1       0.53      0.52      0.53        50
           2       0.56      0.74      0.64        62
           3       0.78      0.60      0.68        42
           4       0.41      0.39      0.40        51

    accuracy                           0.55       245
   macro avg       0.57      0.54      0.55       245
weighted avg       0.56      0.55      0.55       245



In [10]:
print("\nSelect Skin Type:")
for i, val in enumerate(mlb_skin.classes_):
    print(i, ":", val)

skin_choice = int(input("Enter number: "))
user_skin = [mlb_skin.classes_[skin_choice]]


print("\nSelect Skin Condition:")
for i, val in enumerate(mlb_condition.classes_):
    print(i, ":", val)

cond_choice = int(input("Enter number: "))
user_condition = [mlb_condition.classes_[cond_choice]]


print("\nSelect Effect:")
for i, val in enumerate(mlb_effects.classes_):
    print(i, ":", val)

effect_choice = int(input("Enter number: "))
user_effect = [mlb_effects.classes_[effect_choice]]


# ==============================
# 8. CREATE INPUT
# ==============================

user_df = pd.DataFrame(columns=X.columns)
user_df.loc[0] = 0

for val in user_skin + user_condition + user_effect:
    if val in user_df.columns:
        user_df.loc[0, val] = 1


# ==============================
# 9. PREDICT (RANDOM FOREST)
# ==============================

prediction = rf.predict(user_df)
predicted_type = le.inverse_transform(prediction)

print("\n✅ Predicted Product Type:", predicted_type[0])


# ==============================
# 10. SMART RECOMMENDATION
# ==============================

filtered = df[df['product_type'] == predicted_type[0]].copy()

filtered['notable_effects'] = filtered['notable_effects'].astype(str).str.lower()
filtered['skin_condition'] = filtered['skin_condition'].astype(str).str.lower()
filtered['skintype'] = filtered['skintype'].astype(str).str.lower()


def calculate_score(row):
    score = 0

    if user_effect[0] in row['notable_effects']:
        score += 3

    if user_condition[0] in row['skin_condition']:
        score += 2

    if user_skin[0] in row['skintype']:
        score += 1

    return score


filtered['score'] = filtered.apply(calculate_score, axis=1)
filtered = filtered.sort_values(by='score', ascending=False)


# ==============================
# 11. TOP 3 RESULTS
# ==============================

print("\n🔥 Top Recommended Products:\n")

top_results = filtered.head(3)

for i, row in top_results.iterrows():
    print("🔹 Product Name:", row['product_name'])
    print("🔹 Brand:", row['brand'])
    print("🔹 Effects:", ", ".join(clean_text(row['notable_effects'])))
    print("🔹 Score:", row['score'])
    print("🔹 Image URL:", row['picture_src'])
    print("-"*40)


Select Skin Type:
0 : combination
1 : dry
2 : normal
3 : oily
4 : sensitive
Enter number: 4

Select Skin Condition:
0 : acne
1 : dry and dehydrated skin
2 : dull skin
3 : enlarged pores
4 : impaired skin barrier
5 : oily skin
6 : pigmentation
7 : redness
8 : skin imbalance
9 : sun damage
10 : sun protection
11 : wrinkles
Enter number: 3

Select Effect:
0 : acne-free
1 : acne-spot
2 : anti-aging
3 : balancing
4 : black-spot
5 : brightening
6 : hydrating
7 : moisturizing
8 : no-whitecast
9 : oil-control
10 : pore-care
11 : refreshing
12 : skin-barrier
13 : soothing
14 : uv-protection
Enter number: 10

✅ Predicted Product Type: Face Wash

🔥 Top Recommended Products:

🔹 Product Name: BIO=ESSENCE Renew Deep Cleanser 100g
🔹 Brand: BIO ESSENCE
🔹 Effects: refreshing, pore-care
🔹 Score: 6
🔹 Image URL: https://www.beautyhaul.com/assets/uploads/products/thumbs/800x800/jual_bio_essence_renew_deep_cleanser.JPG
----------------------------------------
🔹 Product Name: BIO ESSENCE White Advanced Whit